# DataGuard DeID: PII Detection & Anonymization

Welcome to the **DataGuard DeID** quickstart guide! This library provides robust PII (Personally Identifiable Information) detection, masking, and anonymization specifically tailored for Dutch text, with a focus on Dutch healthcare and health-tech use cases.

> **⚕️ Health-Tech Notice**: This library is designed for the Dutch healthcare sector. Always apply a human-in-the-loop review for critical clinical data, and ensure your full pipeline meets GDPR / AVG and NEN 7510 requirements.

### Key Features
- **19 Entity Types**: Comprehensive coverage including BSN, IBAN, Zorgpolis, Dutch Phone Numbers, etc.
- **Custom Dutch NLP Engine**: spaCy NER + hand-tuned regex recognizers with algorithmic validation.
- **3 Protection Modes**: Anonymize, Tag, and i_tag.
- **Grouped Labels**: Map sub-labels to high-level groups (e.g. `IBAN_CODE` → `FINANCIAL`).
- **Easy Integration**: Simple API for text strings **and** documents (`.pdf`, `.docx`, `.txt`).

### Package Layout
```
dataguard_deid/
├── types.py              — core data structures (RecognizerResult, Pattern, …)
├── analysis/
│   ├── analyzer.py       — PII analysis engine
│   ├── context_awareness.py  — keyword-based score boosting
│   └── overlap_resolver.py   — span deduplication & merging
├── anonymization/
│   ├── engine.py         — stateless anonymization dispatcher
│   └── fake_data.py      — synthetic Dutch PII pools
├── recognizers/
│   ├── base.py           — EntityRecognizer / PatternRecognizer base classes
│   ├── contact.py        — PHONE_NUMBER, EMAIL_ADDRESS, URL
│   ├── datetime.py       — DATE, TIME
│   ├── device.py         — IP_ADDRESS, MAC_ADDRESS, IMEI
│   ├── financial.py      — IBAN_CODE, CREDIT_CARD, CVV
│   ├── identifier.py     — BSN, PASSPORT, ZORGPOLIS_NUMBER
│   ├── location.py       — ZIPCODE, GPS_COORDINATES
│   ├── spacy_recognizer.py  — NER recognizer (PERSON, LOCATION)
│   └── vehicle.py        — LICENCE_PLATE
├── processors/
│   ├── text_processor.py — analyze / guard pipelines for plain-text input
│   └── doc_processor.py  — file reading (.pdf / .docx / .txt) + normalization
├── config/
│   ├── entities.py       — ALL_NL_ENTITY_TYPES list
│   ├── labels.py         — LABEL_GROUPS mapping
│   └── scoring.py        — EntityScoreProfile per entity type
└── patterns/             — Dutch regex patterns & keyword lists
```

### Public Interface
```python
from dataguard_deid import analyze, guard, LABEL_GROUPS

# ── text ──────────────────────────────────────────────
analyze.text("Jan de Vries woont in Amsterdam.")
guard.text("Jan de Vries woont in Amsterdam.")

# ── documents ─────────────────────────────────────────
analyze.doc("/path/to/report.pdf")
guard.doc("/path/to/rapport.docx")
```

## 🛠️ Setup & Installation

First, ensure you have the library and the Dutch spaCy model installed.

In [ ]:
# !pip install dataguard-deid

## 🚀 Basic Usage

Let's start with a simple example of detecting and guarding PII in a Dutch sentence.

In [2]:
from dataguard_deid import analyze, guard

text = """Patiënt Jan Jansen (geboren 15-03-1985, BSN: 200000007) is vandaag gezien.
Contact: jan.jansen@email.nl, Telefoon: +31 6 12345678.
Adres: Herengracht 42, 1015 BN Amsterdam."""

# 1. Analyze — returns a list of finding dicts
findings = analyze.text(text)
print("--- Findings ---")
for f in findings:
    print(f"[{f['type']}] {text[f['start']:f['end']]} (score: {f['score']})")

# 2. Guard — anonymize by default
result = guard.text(text)
print("\n--- Guarded Text (Anonymize) ---")
print(result['guarded_text'])
print("\nFindings in result:", result['findings'])

--- Findings ---
[PERSON] Jan Jansen (score: 0.85)
[DATE] 15-03-1985 (score: 0.95)
[BSN] 200000007 (score: 0.95)
[EMAIL_ADDRESS] jan.jansen@email.nl (score: 1.0)
[PHONE_NUMBER] +31 6 12345678 (score: 1.0)
[ZIPCODE] 1015 BN (score: 0.75)

--- Guarded Text (Anonymize) ---
Patiënt Jan Bakker (geboren 14 januari 1983, BSN: 111222333) is vandaag gezien.
Contact: j.bakker@voorbeeld.nl, Telefoon: +31 6 87654321.
Adres: Herengracht 42, 2500 GH Amsterdam.

Findings in result: [{'type': 'PERSON', 'start': 8, 'end': 18, 'score': 0.85, 'original_text': 'Jan Jansen'}, {'type': 'DATE', 'start': 28, 'end': 38, 'score': 0.95, 'original_text': '15-03-1985'}, {'type': 'BSN', 'start': 45, 'end': 54, 'score': 0.95, 'original_text': '200000007'}, {'type': 'EMAIL_ADDRESS', 'start': 84, 'end': 103, 'score': 1.0, 'original_text': 'jan.jansen@email.nl'}, {'type': 'PHONE_NUMBER', 'start': 115, 'end': 129, 'score': 1.0, 'original_text': '+31 6 12345678'}, {'type': 'ZIPCODE', 'start': 154, 'end': 161, 'score': 0.

## 🛡️ Guard Modes

DataGuard DeID supports three primary modes for handling PII.

In [17]:
modes = ["anonymize", "tag", "i_tag"]
sample = "Bel me op +31 6 1234 5678 of +31 6 8765 4321."

print(f"Original: {sample}\n")
for mode in modes:
    res = guard.text(sample, config={"mode": mode})
    print(f"{mode.capitalize():<10}: {res['guarded_text']}")

Original: Bel me op +31 6 1234 5678 of +31 6 8765 4321.

Anonymize : Bel me op +31 6 23456789 of +31 6 87654321.
Tag       : Bel me op [PHONE_NUMBER] of [PHONE_NUMBER].
I_tag     : Bel me op [PHONE_NUMBER_1] of [PHONE_NUMBER_2].


## 🧪 Advanced: Custom Patterns & Anonymization

You can define your own patterns and even provide pools for synthetic data generation.

In [4]:
from dataguard_deid import custom_pattern

# Define a custom pattern for Employee IDs
emp_pattern = custom_pattern(
    name="EMPLOYEE_ID",
    regex=r"EMP-\d{4}",
    score=0.9,
    context=["medewerker", "employee"],
    anonymize_list=["EMP-9999", "EMP-8888", "EMP-7777"]
)

custom_text = "Medewerker EMP-1234 heeft een verzoek ingediend."

# Analyze with custom pattern
config = {"custom_patterns": [emp_pattern]}
findings = analyze.text(custom_text, config=config)
print("Findings:", findings)

# Guard with custom anonymization pool
guarded = guard.text(custom_text, config=config)
print("Guarded:", guarded['guarded_text'])

Findings: [{'type': 'EMPLOYEE_ID', 'start': 11, 'end': 19, 'score': 1.0}]
Guarded: Medewerker EMP-9999 heeft een verzoek ingediend.


## 🔍 Filtering Entities & Thresholds

Control exactly what is detected and at what confidence level.

In [16]:
config = {
    "set_entities": {
        "ignore": ["PERSON", "LOCATION"],     # Exclude these types
    },
    "score_threshold": 0.6                  # Higher confidence
}

text = "Jan is in Utrecht, zijn BSN is 123456782."
results = analyze.text(text, config=config)
print("Filtered Results (excluding Person/Location):", results)

Filtered Results (only Person/Location): [{'type': 'BSN', 'start': 31, 'end': 40, 'score': 0.95}]


## 🏷️ Grouped Labels

`grouped_labels: True` maps every detected sub-label to a higher-level group.

| Group | Sub-labels |
|-------|-----------|
| `PERSON` | PERSON |
| `DATETIME` | DATE · TIME |
| `CONTACT` | PHONE_NUMBER · EMAIL_ADDRESS · URL |
| `LOCATION` | LOCATION · ZIPCODE · GPS_COORDINATES |
| `FINANCIAL` | IBAN_CODE · CREDIT_CARD · CVV |
| `IDENTIFIER` | BSN · PASSPORT · ZORGPOLIS_NUMBER |
| `DEVICE_IDENTIFIER` | IP_ADDRESS · MAC_ADDRESS · IMEI |
| `VEHICLE_IDENTIFIER` | LICENCE_PLATE |

Each finding gains a `"sub_label"` field with the original entity type, while `"type"` becomes the group name.
In `tag` / `i_tag` modes the bracket labels also use the group name.

In [ ]:
from dataguard_deid import LABEL_GROUPS

sample = """Patiënt Jan Jansen (BSN: 200000007).
IBAN: NL41 RABO 0456 7890 01. IP: 192.168.1.104. Geboortedatum: 15-03-1985."""

# ── analyze with grouped labels ──────────────────────────────────
findings = analyze.text(sample, config={"grouped_labels": True})
print("── Grouped findings ──")
for f in findings:
    print(f"  [{f['type']:<20}] ({f['sub_label']:<18}) {sample[f['start']:f['end']]!r}")

# ── tag mode: bracket tags use the group label ───────────────────
print("\n── Tag mode (grouped) ──")
res_tag = guard.text(sample, config={"grouped_labels": True, "mode": "tag"})
print(res_tag["guarded_text"])

# ── i_tag mode: indexed group labels ─────────────────────────────
print("\n── i_tag mode (grouped) ──")
res_itag = guard.text(sample, config={"grouped_labels": True, "mode": "i_tag"})
print(res_itag["guarded_text"])

# ── inspect the full mapping ──────────────────────────────────────
print("\n── LABEL_GROUPS ──")
for sub, group in LABEL_GROUPS.items():
    print(f"  {sub:<22} → {group}")

## 📄 Document Processing

`analyze.doc()` and `guard.doc()` accept a file **path** instead of a raw string.  
Text is extracted automatically before the PII pipeline runs.

| Format | Reader | Dependency |
|--------|--------|------------|
| `.txt`  | built-in `open()` (UTF-8) | — |
| `.pdf`  | `pypdf` — all pages concatenated | `pip install pypdf` |
| `.docx` | `python-docx` — all paragraphs joined | `pip install python-docx` |

The demo files are a Dutch cardiology discharge letter stored in three formats:
```
examples/files/
├── medisch_verslag.txt
├── medisch_verslag.pdf
└── medisch_verslag.docx
```

All `config` options (modes, filtering, `grouped_labels`) work identically with `.doc()` as with `.text()`.

> **Note:** The extension is validated **before** the file-existence check, so an unsupported format always raises `UnsupportedFormatError` regardless of whether the file exists.

### 📂 Set up file paths

In [6]:
from pathlib import Path

# Notebook lives in examples/ — files are in examples/files/
FILES_DIR = Path("files")

TXT_PATH  = FILES_DIR / "medisch_verslag.txt"
PDF_PATH  = FILES_DIR / "medisch_verslag.pdf"
DOCX_PATH = FILES_DIR / "medisch_verslag.docx"

for p in [TXT_PATH, PDF_PATH, DOCX_PATH]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<30}  {size_kb:6.1f} KB")

  medisch_verslag.txt                1.0 KB
  medisch_verslag.pdf               35.9 KB
  medisch_verslag.docx               7.1 KB


In [7]:
### 📝 .txt — analyze

from dataguard_deid.processors.doc_processor import read as read_doc

# read_doc() gives us the plain text so we can display matched spans
content = read_doc(TXT_PATH)
findings = analyze.doc(TXT_PATH)

print(f"File   : {TXT_PATH.name}")
print(f"Chars  : {len(content)}")
print(f"Findings: {len(findings)}\n")
for f in findings:
    span = content[f['start']:f['end']]
    print(f"  [{f['type']:<18}] {span!r:35}  score={f['score']}")

File   : medisch_verslag.txt
Chars  : 981
Findings: 15

  [PERSON            ] 'Jan-Willem van den Berg'            score=0.85
  [GENDER            ] 'Man'                                score=0.85
  [DATE              ] '12-05-1978'                         score=0.95
  [UNK_NUMBER        ] '138403980'                          score=0.25
  [ZIPCODE           ] '3521 CA'                            score=0.75
  [LOCATION          ] 'Utrecht'                            score=0.85
  [PHONE_NUMBER      ] '+31 6 12345678'                     score=1.0
  [EMAIL_ADDRESS     ] 'j.vandenberg78@gmail.com'           score=1.0
  [DATE              ] '10 maart 2026'                      score=0.85
  [LOCATION          ] 'Vondelpark'                         score=0.85
  [LOCATION          ] 'Amsterdam'                          score=0.85
  [PERSON            ] 'Dr. Anisa Bakker'                   score=0.85
  [IBAN_CODE         ] 'NL91 ABNA 0417 1643 00'             score=0.9
  [URL               ] '

In [8]:
### 📝 .txt — guard (all three modes)

for mode in ["anonymize", "tag", "i_tag"]:
    res = guard.doc(TXT_PATH, config={"mode": mode})
    print(f"{'─'*15} {mode.upper()} {'─'*15}")
    print(res["guarded_text"])

─────────────── ANONYMIZE ───────────────
﻿Medisch Verslag: Cardiologie Overdracht


Patiëntgegevens:


Naam: Maria Janssen
Geslacht: man
Geboortedatum: 27 april 1990
BSN: 598995966 
Adres: Dorpsstraat 45, 2500 GH, Den Haag
Telefoonnummer: +31 6 87654321


E-mail: j.bakker@voorbeeld.nl


Klinische Notitie:
Patiënt is op 14 januari 1983 opgenomen op de afdeling Spoedeisende Hulp wegens aanhoudende pijn op de borst. De patiënt verklaarde dat de klachten begonnen tijdens een fietstocht nabij het Rotterdam in Utrecht.


Tijdens het lichamelijk onderzoek vertoonde de patiënt een verhoogde bloeddruk. Er is een afspraak ingepland voor een vervolgonderzoek bij de specialist, Jan Bakker. Voor de facturatie en administratieve afhandeling is het dossier gekoppeld aan IBAN: NL20 INGB 0001 2345 67.


Behandelplan:
Patiënt dient rust te houden. Voor vragen kan contact worden opgenomen met de assistente via https://www.voorbeeld.nl/pagina of via het directe interne IP-adres van de poli: 10.20.30.40.


### 📰 .pdf — analyze & guard

`pypdf` extracts text page by page; all pages are joined with `\n` before analysis.

In [9]:
# --- analyze ---
content_pdf = read_doc(PDF_PATH)
findings_pdf = analyze.doc(PDF_PATH)

print(f"File    : {PDF_PATH.name}")
print(f"Chars   : {len(content_pdf)}")
print(f"Findings: {len(findings_pdf)}\n")
for f in findings_pdf:
    span = content_pdf[f['start']:f['end']]
    print(f"  [{f['type']:<18}] {span!r:35}  score={f['score']}")

# --- guard (anonymize) ---
print("\n" + "─" * 50)
print("GUARD (anonymize)")
print("─" * 50)
print(guard.doc(PDF_PATH)["guarded_text"])

File    : medisch_verslag.pdf
Chars   : 969
Findings: 15

  [PERSON            ] 'Jan-Willem van den Berg'            score=0.85
  [GENDER            ] 'Man'                                score=0.85
  [DATE              ] '12-05-1978'                         score=0.95
  [UNK_NUMBER        ] '138403980'                          score=0.25
  [ZIPCODE           ] '3521 CA'                            score=0.75
  [LOCATION          ] 'Utrecht'                            score=0.85
  [PHONE_NUMBER      ] '+31 6 12345678'                     score=1.0
  [EMAIL_ADDRESS     ] 'j.vandenberg78@gmail.com'           score=1.0
  [DATE              ] '10 maart 2026'                      score=0.85
  [LOCATION          ] 'Vondelpark'                         score=0.85
  [LOCATION          ] 'Amsterdam'                          score=0.85
  [PERSON            ] 'Dr. Anisa Bakker'                   score=0.85
  [IBAN_CODE         ] 'NL91 ABNA 0417 1643 00'             score=0.9
  [URL               ]

### 📄 .docx — analyze & guard

`python-docx` iterates paragraphs; empty paragraphs become blank lines in the extracted text.

In [10]:
# --- analyze ---
content_docx = read_doc(DOCX_PATH)
findings_docx = analyze.doc(DOCX_PATH)

print(f"File    : {DOCX_PATH.name}")
print(f"Chars   : {len(content_docx)}")
print(f"Findings: {len(findings_docx)}\n")
for f in findings_docx:
    span = content_docx[f['start']:f['end']]
    print(f"  [{f['type']:<18}] {span!r:35}  score={f['score']}")

# --- guard (tag mode) ---
print("\n" + "─" * 50)
print("GUARD (tag)")
print("─" * 50)
print(guard.doc(DOCX_PATH, config={"mode": "tag"})["guarded_text"])

File    : medisch_verslag.docx
Chars   : 975
Findings: 15

  [PERSON            ] 'Jan-Willem van den Berg'            score=0.85
  [GENDER            ] 'Man'                                score=0.85
  [DATE              ] '12-05-1978'                         score=0.95
  [UNK_NUMBER        ] '138403980'                          score=0.25
  [ZIPCODE           ] '3521 CA'                            score=0.75
  [LOCATION          ] 'Utrecht'                            score=0.85
  [PHONE_NUMBER      ] '+31 6 12345678'                     score=1.0
  [EMAIL_ADDRESS     ] 'j.vandenberg78@gmail.com'           score=1.0
  [DATE              ] '10 maart 2026'                      score=0.85
  [LOCATION          ] 'Vondelpark'                         score=0.85
  [LOCATION          ] 'Amsterdam'                          score=0.85
  [PERSON            ] 'Dr. Anisa Bakker'                   score=0.85
  [IBAN_CODE         ] 'NL91 ABNA 0417 1643 00'             score=0.9
  [URL               